In [ ]:
!pip install -q transformers datasets accelerate scikit-learn torch

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc,
    precision_recall_fscore_support
)

from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("/content/go_emotions_treated.csv")

df['BASE_TEXT_PT'] = (
    df['BASE_TEXT_PT']
    .fillna("")        # remove NaN
    .astype(str)       # garante string
)

emotions = [
    'admiration', 'amusement', 'anger', 'annoyance',
    'approval', 'caring', 'confusion', 'curiosity', 'desire',
    'disappointment', 'disapproval', 'disgust', 'embarrassment',
    'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love',
    'nervousness', 'optimism', 'pride', 'realization', 'relief',
    'remorse', 'sadness', 'surprise', 'neutral'
]

num_labels = len(emotions)

print(df.shape)
print("Number of emotions:", num_labels)
df.head(3)

## (1) BERT multilabel sem balanceamento

In [ ]:
X = df["BASE_TEXT_PT"]
y = df[emotions]

X_train, X_aux, y_train, y_aux = train_test_split(
    X, y,
    train_size=0.70,
    random_state=42,
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X_aux, y_aux,
    train_size=0.50,
    random_state=42,
)

df_train = pd.DataFrame({"text": X_train.values, "labels": y_train.values.astype(float).tolist()})
df_dev   = pd.DataFrame({"text": X_dev.values,   "labels": y_dev.values.astype(float).tolist()})
df_test  = pd.DataFrame({"text": X_test.values,  "labels": y_test.values.astype(float).tolist()})

train_ds = Dataset.from_pandas(df_train)
dev_ds   = Dataset.from_pandas(df_dev)
test_ds  = Dataset.from_pandas(df_test)

In [ ]:
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
dev_ds   = dev_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds.set_format("torch")
dev_ds.set_format("torch")
test_ds.set_format("torch")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="samples"
    )

    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        labels, preds, average="micro"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
    }

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

training_args = TrainingArguments(
    output_dir="/content/bertimbau_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    report_to="none",
    fp16=True,
    lr_scheduler_type="linear",
    warmup_ratio=0.2
)

data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

test_results = trainer.evaluate(test_ds)
test_results

## BERT multilabel + CB Loss

In [ ]:
X = df["BASE_TEXT_PT"]
y = df[emotions]

X_train, X_aux, y_train, y_aux = train_test_split(
    X, y,
    train_size=0.70,
    random_state=42,
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X_aux, y_aux,
    train_size=0.50,
    random_state=42,
)

df_train = pd.DataFrame({"text": X_train.values, "labels": y_train.values.tolist()})
df_dev   = pd.DataFrame({"text": X_dev.values,   "labels": y_dev.values.tolist()})
df_test  = pd.DataFrame({"text": X_test.values,  "labels": y_test.values.tolist()})

train_ds = Dataset.from_pandas(df_train)
dev_ds   = Dataset.from_pandas(df_dev)
test_ds  = Dataset.from_pandas(df_test)

In [ ]:
beta = 0.999

samples_per_class = np.sum(y_train.values, axis=0).astype(int)
effective_num = 1.0 - np.power(beta, samples_per_class)
weights = (1.0 - beta) / effective_num
weights = weights / np.sum(weights) * len(weights)

In [ ]:
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        add_special_tokens=True,
        truncation=True,
        max_length=128,
        padding="max_length",
        return_attention_mask=True,
    )

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
dev_ds   = dev_ds.map(tokenize, batched=True, remove_columns=["text"])
test_ds  = test_ds.map(tokenize, batched=True, remove_columns=["text"])

train_ds.set_format("torch")
dev_ds.set_format("torch")
test_ds.set_format("torch")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="samples"
    )

    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        labels, preds, average="micro"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
    }

In [ ]:
num_labels = len(emotions)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

batch_size = 16
num_epochs = 4

steps_per_epoch = len(train_ds) // batch_size
total_training_steps = steps_per_epoch * num_epochs
warmup_steps = int(total_training_steps * 0.2)

class CBTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights).float()

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=0):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        device = logits.device
        labels = labels.float().to(device)

        weights = self.class_weights.to(device)

        weights_for_samples = weights.unsqueeze(0) * labels
        weights_for_samples = weights_for_samples.sum(1)
        weights_for_samples = weights_for_samples.unsqueeze(1)
        weights_for_samples = weights_for_samples.repeat(1, labels.size(1))

        loss = F.binary_cross_entropy_with_logits(
            logits,
            labels,
            weight=weights_for_samples,
            reduction="mean"
        )

        return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/bertimbau_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    report_to="none",
    fp16=True,
    lr_scheduler_type="linear",
    warmup_steps=warmup_steps,
    weight_decay=0.0  # removido para ficar idêntico ao paper
)

trainer = CBTrainer(
    class_weights=weights,
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

predictions = trainer.predict(test_ds)

logits = predictions.predictions
y_true = np.array(predictions.label_ids)

y_prob = 1 / (1 + np.exp(-logits))  # sigmoid
y_pred = (y_prob > 0.5).astype(int)

print("\n===== MÉTRICAS GLOBAIS =====")
print({
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision (Macro)": precision_score(y_true, y_pred, average="macro"),
    "Recall (Macro)": recall_score(y_true, y_pred, average="macro"),
    "F1 (Macro)": f1_score(y_true, y_pred, average="macro"),
})

per_label_metrics = []

for i, label_name in enumerate(emotions):
    acc = accuracy_score(y_true[:, i], y_pred[:, i])
    prec = precision_score(y_true[:, i], y_pred[:, i])
    rec = recall_score(y_true[:, i], y_pred[:, i])
    f1 = f1_score(y_true[:, i], y_pred[:, i])

    per_label_metrics.append({
        "Emotion": label_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })

per_label_df = pd.DataFrame(per_label_metrics)

print("\n===== MÉTRICAS POR EMOÇÃO (BERTimbau) =====")
print(per_label_df.round(2))